# Data Cleaning
This notebook handles cleaning the filtered pain medication data, including handling missing values, standardizing text, removing duplicates, and detecting outliers.

## 1. Import Required Libraries

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import os

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

## 2. Load Filtered Data

In [ ]:
# TODO: Load filtered data
filtered_data_path = '../../data/filtered/pain_meds_filtered.csv'

df = pd.read_csv(filtered_data_path)

print(f"Filtered data loaded successfully!")
print(f"Shape: {df.shape}")
print(f"\nFirst few rows:")
display(df.head())

## 3. Check Missing Values

In [ ]:
# TODO: Check for missing values
print("Missing Values Summary:")
print("=" * 60)

missing_counts = df.isnull().sum()
missing_percentages = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Column': missing_counts.index,
    'Missing Count': missing_counts.values,
    'Missing Percentage': missing_percentages.values
})

display(missing_df[missing_df['Missing Count'] > 0])

if missing_df['Missing Count'].sum() == 0:
    print("\nNo missing values found!")
else:
    print(f"\nTotal missing values: {missing_df['Missing Count'].sum()}")

## 4. Handle Missing Values

In [ ]:
# TODO: Handle missing values
# For numeric columns: use median
# For categorical columns: use mode or drop

df_cleaned = df.copy()

# Handle numeric columns with median
numeric_columns = df_cleaned.select_dtypes(include=[np.number]).columns
for col in numeric_columns:
    if df_cleaned[col].isnull().sum() > 0:
        median_value = df_cleaned[col].median()
        df_cleaned[col].fillna(median_value, inplace=True)
        print(f"Filled {col} with median: {median_value}")

# Handle categorical columns with mode or drop
categorical_columns = df_cleaned.select_dtypes(include=['object']).columns
for col in categorical_columns:
    if df_cleaned[col].isnull().sum() > 0:
        # Drop rows with missing categorical values if less than 5% of data
        missing_pct = (df_cleaned[col].isnull().sum() / len(df_cleaned)) * 100
        if missing_pct < 5:
            df_cleaned = df_cleaned.dropna(subset=[col])
            print(f"Dropped rows with missing {col} ({missing_pct:.2f}%)")
        else:
            mode_value = df_cleaned[col].mode()[0]
            df_cleaned[col].fillna(mode_value, inplace=True)
            print(f"Filled {col} with mode: {mode_value}")

print(f"\nRows after handling missing values: {len(df_cleaned)}")

## 5. Standardize Text in Condition Column

In [ ]:
# TODO: Standardize text - lowercase and strip whitespace
print("Standardizing condition column...")

# Convert to lowercase and strip whitespace
df_cleaned['condition'] = df_cleaned['condition'].str.lower().str.strip()

# Also standardize drugName column
df_cleaned['drugName'] = df_cleaned['drugName'].str.strip()

print("Text standardization complete!")
print(f"\nUnique conditions: {df_cleaned['condition'].nunique()}")
print(f"Unique drugs: {df_cleaned['drugName'].nunique()}")

## 6. Remove Duplicates

In [ ]:
# TODO: Remove duplicate rows
print(f"Rows before removing duplicates: {len(df_cleaned)}")

# Check for duplicates
duplicates = df_cleaned.duplicated().sum()
print(f"Duplicate rows found: {duplicates}")

# Remove duplicates
df_cleaned = df_cleaned.drop_duplicates()

print(f"Rows after removing duplicates: {len(df_cleaned)}")
print(f"Rows removed: {duplicates}")

## 7. IQR Outlier Detection on Rating Column

In [ ]:
# TODO: Detect outliers using IQR method
# Note: For rating column (typically 1-10), outliers may not be an issue
# This is more relevant for other numeric columns if present

if 'rating' in df_cleaned.columns:
    print("Detecting outliers in rating column using IQR method...")
    
    Q1 = df_cleaned['rating'].quantile(0.25)
    Q3 = df_cleaned['rating'].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    print(f"Q1: {Q1}, Q3: {Q3}, IQR: {IQR}")
    print(f"Lower bound: {lower_bound}, Upper bound: {upper_bound}")
    
    # Identify outliers
    outliers = df_cleaned[(df_cleaned['rating'] < lower_bound) | (df_cleaned['rating'] > upper_bound)]
    print(f"\nOutliers detected: {len(outliers)}")
    
    # For ratings, we typically don't remove outliers as they represent valid data
    # But we document them for awareness
    if len(outliers) > 0:
        print(f"\nOutlier rating distribution:")
        print(outliers['rating'].value_counts().sort_index())
else:
    print("Rating column not found in dataset.")

## 8. Save Cleaned Data

In [ ]:
# TODO: Save cleaned data to CSV
output_path = '../../data/cleaned/pain_meds_cleaned.csv'

# Create directory if it doesn't exist
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# Save to CSV
df_cleaned.to_csv(output_path, index=False)

print(f"Cleaned data saved to: {output_path}")
print("\n" + "=" * 60)
print("DATA CLEANING SUMMARY")
print("=" * 60)
print(f"Original rows: {len(df):,}")
print(f"Cleaned rows: {len(df_cleaned):,}")
print(f"Rows removed: {len(df) - len(df_cleaned):,}")
print(f"Columns: {len(df_cleaned.columns)}")
print(f"Missing values remaining: {df_cleaned.isnull().sum().sum()}")
print("=" * 60)